# 01 - Data Cleaning & Validation

Wejście: `data/raw/Hotels.csv` · Wyjście: `data/processed/hotels_clean.parquet` + `data/sample/hotels_sample.csv`

**TODO:**
- konwersja serial Excela -> daty (Checkin/Checkout)
- obsługa ujemnych `Net_Revenue` / `GST` (anulacje)
- typy, sanity checks, walidacja
- zapis czystego pliku

In [204]:
import pandas as pd
import numpy as np
from pathlib import Path

In [205]:
CWD = Path.cwd()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "Hotels.csv"

In [206]:
df = pd.read_csv(DATA_PATH)
df.head()
# Checkin and checkout are intigers, but they should be datetime


,Booking_ID,Hotel_Name,Hotel_Category,City,State,Country,Checkin_Date,Checkout_Date,Month,Quarter,...,Net_Revenue,GST_Amount,Total_With_GST,Booking_Channel,Customer_Type,Payment_Mode,Occupancy_Status,Customer_Rating,Feedback_Status,Covid_Impact_Level
0,BK9835534,ITC Kakatiya,Luxury,Hyderabad,Telangana,India,44320,44324,May,Q2,...,89160,10699.20,99859.20,Walk-in,Regular,UPI,Checked-Out,4.6,Positive,High
1,BK7803377,ITC Royal Bengal,Luxury,Kolkata,West Bengal,India,44352,44356,June,Q2,...,86830,10419.60,97249.60,Corporate,VIP,Credit Card,Checked-In,4.6,Positive,High
2,BK3686278,ITC Grand Goa,Resort,Goa,Goa,India,44443,44445,September,Q3,...,29389,3526.68,32915.68,Booking.com,Corporate,Cash,Checked-In,4.2,Positive,Low
3,BK7514886,Fortune Select Trinity,Business,Bengaluru,Karnataka,India,44469,44470,September,Q3,...,24409,2929.08,27338.08,Website,Regular,Debit Card,Cancelled,3.8,Neutral,Low
4,BK6312815,ITC Narmada,Business,Ahmedabad,Gujarat,India,44461,44463,September,Q3,...,9940,1192.80,11132.80,Walk-in,New Guest,UPI,Checked-In,4.6,Negative,Low


In [207]:
df.info()
df.describe()
df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 300000 entries, 0 to 299999
Data columns (total 28 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   Booking_ID          300000 non-null  str    
 1   Hotel_Name          300000 non-null  str    
 2   Hotel_Category      300000 non-null  str    
 3   City                300000 non-null  str    
 4   State               300000 non-null  str    
 5   Country             300000 non-null  str    
 6   Checkin_Date        300000 non-null  int64  
 7   Checkout_Date       300000 non-null  int64  
 8   Month               300000 non-null  str    
 9   Quarter             300000 non-null  str    
 10  Year                300000 non-null  int64  
 11  Room_Type           300000 non-null  str    
 12  Guests              300000 non-null  int64  
 13  Nights_Stayed       300000 non-null  int64  
 14  Room_Rate           300000 non-null  int64  
 15  Extra_Charges       300000 non-null  int64  


Booking_ID            0
Hotel_Name            0
Hotel_Category        0
City                  0
State                 0
Country               0
Checkin_Date          0
Checkout_Date         0
Month                 0
Quarter               0
Year                  0
Room_Type             0
Guests                0
Nights_Stayed         0
Room_Rate             0
Extra_Charges         0
Discount              0
Gross_Revenue         0
Net_Revenue           0
GST_Amount            0
Total_With_GST        0
Booking_Channel       0
Customer_Type         0
Payment_Mode          0
Occupancy_Status      0
Customer_Rating       0
Feedback_Status       0
Covid_Impact_Level    0
dtype: int64

In [208]:
df['Checkin_Date_time'] = pd.to_datetime(df['Checkin_Date'], unit='D', origin='1899-12-30')
df['Checkout_Date_time'] = pd.to_datetime(df['Checkout_Date'], unit='D', origin='1899-12-30')
df.head()

,Booking_ID,Hotel_Name,Hotel_Category,City,State,Country,Checkin_Date,Checkout_Date,Month,Quarter,...,Total_With_GST,Booking_Channel,Customer_Type,Payment_Mode,Occupancy_Status,Customer_Rating,Feedback_Status,Covid_Impact_Level,Checkin_Date_time,Checkout_Date_time
0,BK9835534,ITC Kakatiya,Luxury,Hyderabad,Telangana,India,44320,44324,May,Q2,...,99859.20,Walk-in,Regular,UPI,Checked-Out,4.6,Positive,High,2021-05-04,2021-05-08
1,BK7803377,ITC Royal Bengal,Luxury,Kolkata,West Bengal,India,44352,44356,June,Q2,...,97249.60,Corporate,VIP,Credit Card,Checked-In,4.6,Positive,High,2021-06-05,2021-06-09
2,BK3686278,ITC Grand Goa,Resort,Goa,Goa,India,44443,44445,September,Q3,...,32915.68,Booking.com,Corporate,Cash,Checked-In,4.2,Positive,Low,2021-09-04,2021-09-06
3,BK7514886,Fortune Select Trinity,Business,Bengaluru,Karnataka,India,44469,44470,September,Q3,...,27338.08,Website,Regular,Debit Card,Cancelled,3.8,Neutral,Low,2021-09-30,2021-10-01
4,BK6312815,ITC Narmada,Business,Ahmedabad,Gujarat,India,44461,44463,September,Q3,...,11132.80,Walk-in,New Guest,UPI,Checked-In,4.6,Negative,Low,2021-09-22,2021-09-24


In [209]:
print(df['Checkin_Date_time'].max(), df['Checkin_Date_time'].min())
print(df['Checkout_Date_time'].max(), df['Checkout_Date_time'].min())
# Checkin and checkout dates are between 2021 and 2022, which is good.

2022-03-31 00:00:00 2021-04-01 00:00:00
2022-04-05 00:00:00 2021-04-02 00:00:00


In [210]:
df['nights_calc'] = (df['Checkout_Date_time'] - df['Checkin_Date_time']).dt.days
print(df.head())
print('Checkout < Checkin:', (df['Checkout_Date_time'] < df['Checkin_Date_time']).sum())
print('nights_calc != Nights_Stayed:', (df['nights_calc'] != df['Nights_Stayed']).sum())
# Nights_Stayed and nights_calc are the same, which is good. There are no cases where checkout is before checkin, which is also good.

  Booking_ID              Hotel_Name Hotel_Category       City        State  \
0  BK9835534            ITC Kakatiya         Luxury  Hyderabad    Telangana   
1  BK7803377        ITC Royal Bengal         Luxury    Kolkata  West Bengal   
2  BK3686278           ITC Grand Goa         Resort        Goa          Goa   
3  BK7514886  Fortune Select Trinity       Business  Bengaluru    Karnataka   
4  BK6312815             ITC Narmada       Business  Ahmedabad      Gujarat   

  Country  Checkin_Date  Checkout_Date      Month Quarter  ...  \
0   India         44320          44324        May      Q2  ...   
1   India         44352          44356       June      Q2  ...   
2   India         44443          44445  September      Q3  ...   
3   India         44469          44470  September      Q3  ...   
4   India         44461          44463  September      Q3  ...   

   Booking_Channel Customer_Type  Payment_Mode  Occupancy_Status  \
0          Walk-in       Regular           UPI       Checked

In [211]:
print('Year incorrect:', (df['Year'] != df['Checkin_Date_time'].dt.year).sum())
print('Month incorrect:',(df['Month'] != df['Checkin_Date_time'].dt.strftime('%B')).sum())
# Maintaining data consistency and accurate documentation 

Year incorrect: 0
Month incorrect: 0


In [212]:
print(df['Gross_Revenue'].describe())
print(df['Net_Revenue'].isnull().sum())
print(df['Net_Revenue'].describe())


count    300000.000000
mean      52216.514390
std       32220.392052
min        4581.000000
25%       26118.000000
50%       44941.500000
75%       73198.000000
max      146886.000000
Name: Gross_Revenue, dtype: float64
0
count    300000.000000
mean      49716.107807
std       32255.358866
min        -211.000000
25%       23599.000000
50%       42477.500000
75%       70699.000000
max      145752.000000
Name: Net_Revenue, dtype: float64


In [213]:
revenue_check = pd.crosstab(index=df['Occupancy_Status'], columns=df['Net_Revenue'] <= 0)
print(revenue_check)

Net_Revenue        False  True 
Occupancy_Status               
Cancelled          29698      0
Checked-In        150158      2
Checked-Out       105215      0
No Show            14927      0


In [214]:
revenue_check_net_rev = df[df['Net_Revenue']<= 0]
print(revenue_check_net_rev)

       Booking_ID     Hotel_Name Hotel_Category  City          State Country  \
63092   BK9465293     ITC Mughal         Resort  Agra  Uttar Pradesh   India   
128401  BK3355219  ITC Grand Goa         Resort   Goa            Goa   India   

        Checkin_Date  Checkout_Date     Month Quarter  ...  Booking_Channel  \
63092          44473          44474   October      Q4  ...       MakeMyTrip   
128401         44616          44617  February      Q1  ...          Website   

       Customer_Type  Payment_Mode  Occupancy_Status  Customer_Rating  \
63092         Member   Credit Card        Checked-In              4.6   
128401     Corporate   Net Banking        Checked-In              3.5   

        Feedback_Status  Covid_Impact_Level  Checkin_Date_time  \
63092           Neutral                 Low         2021-10-04   
128401         Positive                 Low         2022-02-24   

        Checkout_Date_time  nights_calc  
63092           2021-10-05            1  
128401          20

In [215]:
    Net_gst = pd.crosstab(index=df['Occupancy_Status'], columns=df['GST_Amount'] <= 0)
    print(Net_gst)
    print(df[df['GST_Amount'] <= 0])

GST_Amount         False  True 
Occupancy_Status               
Cancelled          29698      0
Checked-In        150158      2
Checked-Out       105215      0
No Show            14927      0
       Booking_ID     Hotel_Name Hotel_Category  City          State Country  \
63092   BK9465293     ITC Mughal         Resort  Agra  Uttar Pradesh   India   
128401  BK3355219  ITC Grand Goa         Resort   Goa            Goa   India   

        Checkin_Date  Checkout_Date     Month Quarter  ...  Booking_Channel  \
63092          44473          44474   October      Q4  ...       MakeMyTrip   
128401         44616          44617  February      Q1  ...          Website   

       Customer_Type  Payment_Mode  Occupancy_Status  Customer_Rating  \
63092         Member   Credit Card        Checked-In              4.6   
128401     Corporate   Net Banking        Checked-In              3.5   

        Feedback_Status  Covid_Impact_Level  Checkin_Date_time  \
63092           Neutral                 Low

In [216]:
Total_gst = pd.crosstab(index=df['Occupancy_Status'], columns=df['Total_With_GST'] <= 0)
print(Total_gst)
print(df[df['Total_With_GST'] <= 0])

Total_With_GST     False  True 
Occupancy_Status               
Cancelled          29698      0
Checked-In        150158      2
Checked-Out       105215      0
No Show            14927      0
       Booking_ID     Hotel_Name Hotel_Category  City          State Country  \
63092   BK9465293     ITC Mughal         Resort  Agra  Uttar Pradesh   India   
128401  BK3355219  ITC Grand Goa         Resort   Goa            Goa   India   

        Checkin_Date  Checkout_Date     Month Quarter  ...  Booking_Channel  \
63092          44473          44474   October      Q4  ...       MakeMyTrip   
128401         44616          44617  February      Q1  ...          Website   

       Customer_Type  Payment_Mode  Occupancy_Status  Customer_Rating  \
63092         Member   Credit Card        Checked-In              4.6   
128401     Corporate   Net Banking        Checked-In              3.5   

        Feedback_Status  Covid_Impact_Level  Checkin_Date_time  \
63092           Neutral                 Low

# Delete 2 rows because the data indicates that 2 people are checked in and checked out at the same time

In [217]:
values_to_drop = ['BK9465293', 'BK3355219']
df = df[~df['Booking_ID'].isin(values_to_drop)]
print(df.shape)

(299998, 31)


In [218]:
# 1. Tworzymy mniejszy, roboczy DataFrame tylko z interesującymi nas kolumnami finansowymi
check_df = df[['Room_Rate', 'Nights_Stayed', 'Extra_Charges', 'Gross_Revenue', 'Discount', 'Net_Revenue', 'GST_Amount', 'Total_With_GST']].copy()

check_df['Calc_Gross'] = (check_df['Room_Rate'] * check_df['Nights_Stayed']) + check_df['Extra_Charges']
check_df['Calc_Net'] = check_df['Gross_Revenue'] - check_df['Discount']
check_df['Calc_Total'] = check_df['Net_Revenue'] + check_df['GST_Amount']

display(check_df[['Gross_Revenue', 'Calc_Gross', 'Net_Revenue', 'Calc_Net', 'Total_With_GST', 'Calc_Total']].head(10))

mismatch_gross = (check_df['Calc_Gross'] - check_df['Gross_Revenue']).abs() > 1
mismatch_net = (check_df['Calc_Net'] - check_df['Net_Revenue']).abs() > 1
mismatch_total = (check_df['Calc_Total'] - check_df['Total_With_GST']).abs() > 1

bad_rows = check_df[mismatch_gross | mismatch_net | mismatch_total]
print(f"Found {len(bad_rows)} rows that fail the consistency test.")

,Gross_Revenue,Calc_Gross,Net_Revenue,Calc_Net,Total_With_GST,Calc_Total
0,92666,92666,89160,89160,99859.20,99859.20
1,91043,91043,86830,86830,97249.60,97249.60
2,33191,33191,29389,29389,32915.68,32915.68
3,27690,27690,24409,24409,27338.08,27338.08
4,14556,14556,9940,9940,11132.80,11132.80
5,29665,29665,24977,24977,27974.24,27974.24
6,105755,105755,105459,105459,118114.08,118114.08
7,18035,18035,15926,15926,17837.12,17837.12
8,112410,112410,110670,110670,123950.40,123950.40
9,60315,60315,58649,58649,65686.88,65686.88


Found 0 rows that fail the consistency test.


Gross = Room_Rate × Nights_Stayed + Extra_Charges
Net = Gross − Discount
Total = Net + GST

In [219]:
# 1. Izolujemy wszystkie wiersze z powtarzającym się Booking_ID
duplicates = df[df.duplicated(subset=['Booking_ID'], keep=False)].copy()

# 2. Sortujemy, żeby widzieć te same ID jedno pod drugim
duplicates = duplicates.sort_values(by='Booking_ID')

columns_to_view = ['Booking_ID', 'Hotel_Category', 'Customer_Type', 'Room_Type','Checkin_Date_time', 'Checkout_Date_time', 'Nights_Stayed'] 
# 3. dodano dodatkowo kolumny dat 

display(duplicates[columns_to_view].head(100))

,Booking_ID,Hotel_Category,Customer_Type,Room_Type,Checkin_Date_time,Checkout_Date_time,Nights_Stayed
7891,BK1000640,Luxury,Regular,Deluxe,2021-09-26,2021-10-01,5
133974,BK1000640,Business,VIP,Presidential Suite,2021-07-12,2021-07-14,2
56406,BK1001335,Business,New Guest,Presidential Suite,2022-01-04,2022-01-06,2
252288,BK1001335,Luxury,Regular,Presidential Suite,2021-06-01,2021-06-05,4
24155,BK1006611,Business,Member,Executive,2022-02-12,2022-02-14,2
...,...,...,...,...,...,...,...
57896,BK1111348,Business,Regular,Luxury Suite,2022-02-08,2022-02-12,4
101631,BK1111348,Luxury,Member,Deluxe,2021-07-31,2021-08-05,5
46567,BK1113150,Resort,Member,Club Room,2021-12-19,2021-12-22,3
194138,BK1113150,Business,New Guest,Presidential Suite,2021-05-14,2021-05-18,4


In [220]:

validation = duplicates.groupby('Booking_ID').agg(
    unique_hotels=('Hotel_Category', 'nunique'),
    unique_customers=('Customer_Type', 'nunique')
)

# Hypothesis 1: Key Error
# Different hotels or customers were incorrectly assigned the same ID.
key_errors = validation[(validation['unique_hotels'] > 1) | (validation['unique_customers'] > 1)]

print(f"Number of Booking_IDs classified as key errors: {len(key_errors)}")


Number of Booking_IDs classified as key errors: 4490


Booking_ID is not a unique identifier. The data shows different hotels, cities, and dates sharing the exact same ID. This is caused by key collisions from integrating 15 independent booking systems. Since these are valid records, no rows will be deleted. Moving forward, each row must be treated as a separate transaction.

In [221]:
# 1. Usunięcie surowych serialnych kolumn z datami
columns_to_drop = ['Checkin_Date', 'Checkout_Date']
df = df.drop(columns=columns_to_drop)

# 2. Zmiana nazw kolumn z datami na docelowe
df = df.rename(columns={
    'Checkin_Date_time': 'Checkin_Date',
    'Checkout_Date_time': 'Checkout_Date'
})

# 3. Optymalizacja pamięci i konwersja do kategorii
mem_before = df.memory_usage(deep=True).sum() / (1024 ** 2)

cat_cols = [
    'Hotel_Category', 'Booking_Channel', 'Customer_Type', 
    'Occupancy_Status', 'Room_Type', 'Payment_Mode', 
    'Feedback_Status', 'Covid_Impact_Level', 'City', 'State'
]
existing_cat_cols = [col for col in cat_cols if col in df.columns]
df[existing_cat_cols] = df[existing_cat_cols].astype('category')

mem_after = df.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"Optymalizacja pamięci: {mem_before:.2f} MB -> {mem_after:.2f} MB | Zysk: {(mem_before - mem_after):.2f} MB ({((mem_before - mem_after)/mem_before)*100:.1f}%)")


# 4. Zapis do Parquet oparty o PROJECT_ROOT (przy użyciu pathlib)
processed_dir = PROJECT_ROOT / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True) 

output_path = processed_dir / 'hotels_clean.parquet'

df.to_parquet(output_path, index=False)
print(f"\nGotowy do analizy plik zapisano w: {output_path}")

Optymalizacja pamięci: 101.57 MB -> 58.85 MB | Zysk: 42.72 MB (42.1%)

Gotowy do analizy plik zapisano w: /Users/maksym/projects/hotel-revenue-analytics/data/processed/hotels_clean.parquet


### English 

### 🧹 Data Cleaning & Preparation Summary

**Row Count Tracking:**
* **Initial records:** 300,000
* **Final records:** 299,998 (2 anomalous rows dropped during validation)

**Key Decisions & Transformations:**
1. **Revenue Formula Verification:** Confirmed exact formula relationships at the row level (`Gross_Revenue = Room_Rate * Nights_Stayed + Extra_Charges`, `Net_Revenue = Gross_Revenue - Discount`, `Total_With_GST = Net_Revenue + GST_Amount`).
2. **Booking_ID Uniqueness:** Confirmed that `Booking_ID` is not a unique primary key. The data shows different hotels, cities, and dates sharing the exact same ID, which indicates key collisions from integrating 15 independent booking systems. Since these are valid records, no rows were deleted. Moving forward, each row must be treated as a separate transaction.
3. **Date Standardization:** Dropped redundant Excel serial date columns to maintain a clean dataset, keeping only proper `datetime` objects and renaming them to `Checkin_Date` and `Checkout_Date`.
4. **Memory Optimization:** Converted low-cardinality text columns to `category` data types. This operation reduced RAM usage by 42.1% (from ~101.5 MB to ~58.8 MB) and prepared the dataset for faster analytical aggregations.

**Output:** The cleaned dataset has been exported to `data/processed/hotels_clean.parquet` with preserved metadata and data types, fully prepared for Exploratory Data Analysis (EDA) and SQL querying.

---

### Polish 

### 🧹 Podsumowanie czyszczenia i przygotowania danych

**Zmiany w liczbie wierszy:**
* **Początkowa liczba rekordów:** 300 000
* **Końcowa liczba rekordów:** 299 998 (usunięto 2 anomalne wiersze podczas walidacji)

**Kluczowe decyzje i transformacje:**
1. **Weryfikacja spójności wyliczeń przychodu:** Potwierdzono dokładne relacje matematyczne na poziomie każdego wiersza (`Gross_Revenue = Room_Rate * Nights_Stayed + Extra_Charges`, `Net_Revenue = Gross_Revenue - Discount`, `Total_With_GST = Net_Revenue + GST_Amount`).
2. **Brak unikalności Booking_ID:** Potwierdzono, że `Booking_ID` nie jest unikalnym kluczem głównym. Dane pokazują, że zupełnie inne hotele, miasta i daty współdzielą to samo ID, co jest wynikiem kolizji kluczy przy integracji 15 niezależnych systemów rezerwacyjnych. Ponieważ są to prawidłowe rekordy, nie usunięto żadnych wierszy. W dalszych krokach każdy wiersz musi być traktowany jako odrębna transakcja.
3. **Standaryzacja dat:** Usunięto redundantne kolumny z surowymi serialami dat z Excela, aby utrzymać czystość zbioru, pozostawiając wyłącznie prawidłowe obiekty `datetime` i zmieniając ich nazwy docelowe na `Checkin_Date` i `Checkout_Date`.
4. **Optymalizacja pamięci:** Skonwertowano kolumny tekstowe o małej liczbie unikalnych wartości na typ kategoryczny (`category`). Operacja ta zmniejszyła zużycie pamięci RAM o 42,1% (z ok. 101,5 MB na ok. 58,8 MB) i przygotowała zbiór do szybszych agregacji analitycznych.

**Wynik końcowy:** Oczyszczony zbiór danych został wyeksportowany do `data/processed/hotels_clean.parquet` z zachowaniem metadanych i typów danych, w pełni gotowy do etapu EDA (Eksploracyjnej Analizy Danych) oraz odpytywania w SQL.